In [ ]:
import random
import os
import numpy as np
import time
import torch
import torch.nn as nn 
import pandas as pd 
import argparse
import torch.optim as optim
import torch.utils.data as data
from tensorboardX import SummaryWriter

In [ ]:
DATA_URL = "http://files.grouplens.org/datasets/movielens/ml-100k/u.data"

MAIN_PATH = r'D:\JupyterCode\NCF-moocdata'
DATA_PATH = MAIN_PATH + '\\data\\mooc_data\\filtered.csv'
MODEL_PATH = MAIN_PATH + '\\models\\'

MODEL = 'mooc_Neu_MF'

载入数据：

In [ ]:
# load data
ml_1m = pd.read_csv(
    DATA_PATH, 
    sep=",", 
    names = ['user_id', 'item_id', 'rating', 'timestamp'], 
    engine='python')

参数设置：

In [ ]:
#设置伪随机数，方便实验结果复现
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 即使没有 CUDA，也可以设置，不会产生影响
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # 在没有 CUDA 的情况下通常将其设置为 False

In [ ]:
parser = argparse.ArgumentParser() 
parser.add_argument("--seed", type=int, default=42,help="Seed")
parser.add_argument("--lr", type=float, default=0.001, help="learning rate")
parser.add_argument("--dropout",type=float,default=0.2, help="dropout rate")
parser.add_argument("--batch_size", type=int, default=256,help="batch size for training")
parser.add_argument("--epochs", type=int,default=30,  help="training epoches")
parser.add_argument("--top_k", type=int, default=10, help="compute metrics@top_k")
# 用于计算评估指标的top-k值，通常用于评估模型的性能。默认值为10。
parser.add_argument("--factor_num", type=int,default=32, help="predictive factors numbers in the model")
# 模型中的潜在因子数量，通常用于嵌入层的维度。默认值为32
parser.add_argument("--layers",nargs='+', default=[64,32,16,8], help="MLP layers. Note that the first layer is the concatenation of user and item embeddings. So layers[0]/2 is the embedding size.")
# MLP（多层感知器）的层次结构，即神经网络中的隐藏层。默认设置为[64, 32, 16, 8]，表示模型包含4个隐藏层，每层的神经元数量分别是64、32、16和8。
parser.add_argument("--num_ng", type=int,default=4, help="Number of negative samples for training set")
# 用于训练集的负样本数量，即每个正样本对应的负样本数量。默认值为4。
parser.add_argument("--num_ng_test", type=int,default=100, help="Number of negative samples for test set")
# 用于测试集的负样本数量，即每个正样本对应的负样本数量。默认值为100。
parser.add_argument("--out", default=True,help="save model or not")
# 用于测试集的负样本数量，即每个正样本对应的负样本数量。默认值为100。

# set device and parameters
# args = parser.parse_args()有问题解决看： https://blog.csdn.net/wmq104/article/details/123534597
args=parser.parse_known_args()[0]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter()#将各种信息写入 TensorBoard 日志文件，以便在 TensorBoard 界面中查看和分析实验结果。

# seed for Reproducibility
# util.seed_everything(args.seed)
seed_everything(args.seed)

下面是 argparse.ArgumentParser() 函数的详细解释：
parser = argparse.ArgumentParser(
    description='Program Description',
    epilog='Program Epilog'
)
description（可选参数）：用于指定程序的简短描述或概述，通常会在用户运行 --help 或 -h 选项时显示在帮助文档中。
epilog（可选参数）：用于指定程序的结尾说明，通常会在用户运行 --help 或 -h 选项时显示在帮助文档中。
argparse.ArgumentParser() 创建了一个 ArgumentParser 对象，你可以使用它来定义和解析命令行参数。

可以使用 add_argument() 方法向 ArgumentParser 对象添加命令行参数的定义。
--lr：是命令行参数的名称，以 -- 开头表示这是一个长选项。
type=float：指定参数的类型，这里是浮点数。
default=0.001：指定参数的默认值。
help：提供了关于参数的帮助文本，通常在用户运行 --help 或 -h 选项时显示。
添加完所有参数的定义后，你可以使用 parse_args() 方法来解析命令行参数，并将它们存储在一个 Namespace 对象中。
你可以通过 args 对象来访问命令行参数的值，例如 args.lr 来获取学习率的值。

处理数据：

In [ ]:
"""
这个自定义数据集类的主要目的是将原始数据组织成适用于 PyTorch 模型训练和测试的形式。
通过实现这些方法，可以方便地使用 PyTorch 提供的数据加载器来加载和处理数据，以供深度学习模型使用。
"""
class Rating_Datset(torch.utils.data.Dataset):
    def __init__(self, user_list, item_list, rating_list):
        super(Rating_Datset, self).__init__()
        self.user_list = user_list
        self.item_list = item_list
        self.rating_list = rating_list

    def __len__(self):
        return len(self.user_list)

    def __getitem__(self, idx):
        user = self.user_list[idx]
        item = self.item_list[idx]
        rating = self.rating_list[idx]

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(item, dtype=torch.long),
            torch.tensor(rating, dtype=torch.float)
            )

In [ ]:
class NCF_Data(object):
    """
    Construct Dataset for NCF
    """
    def __init__(self, args, ratings):
        self.ratings = ratings
        self.num_ng = args.num_ng
        self.num_ng_test = args.num_ng_test
        self.batch_size = args.batch_size

        self.preprocess_ratings = self._reindex(self.ratings)

        self.user_pool = set(self.ratings['user_id'].unique())
        self.item_pool = set(self.ratings['item_id'].unique())

        self.train_ratings, self.test_ratings = self._leave_one_out(self.preprocess_ratings)
        self.negatives = self._negative_sampling(self.preprocess_ratings)
        random.seed(args.seed)

        
    """
    本函数作用：
    1.用整数重新索引
    2.将有无交互映射为0，1

    """   
    def _reindex(self, ratings):
        """
        Process dataset to reindex userID and itemID, also set rating as binary feedback
        """
        user_list = list(ratings['user_id'].drop_duplicates())
        user2id = {w: i for i, w in enumerate(user_list)}
        item_list = list(ratings['item_id'].drop_duplicates())
        item2id = {w: i for i, w in enumerate(item_list)}
        #  enumerate(item_list) 是一个迭代器，用于遍历 item_list 列表中的元素。
        # 在每次迭代中，enumerate 返回一个元组，其中第一个元素 i 是元素的索引（从0开始），第二个元素 w 是元素的值（即物品ID）。
        ratings['user_id'] = ratings['user_id'].apply(lambda x: user2id[x])
        ratings['item_id'] = ratings['item_id'].apply(lambda x: item2id[x])
        # .apply() 方法的作用，ratings['item_id'] 列中的每个原始物品ID都被替换为了相应的整数ID，完成了物品ID的重新索引
        ratings['rating'] = ratings['rating'].apply(lambda x: float(x > 0))
        # .apply() 方法的作用，ratings['rating'] 列中的每个原始评分值都被替换为了相应的二进制反馈值。
        return ratings
                                             
                                                
    """
    这种 "leave-one-out" 评估协议的核心思想是将每个用户的最新交互数据作为测试集，其余数据作为训练集，以评估模型的性能。
    这有助于模拟实际应用场景中的推荐任务，其中用户的历史行为用于训练模型，最新行为用于测试模型。

    """
    def _leave_one_out(self, ratings):
        """
        leave-one-out evaluation protocol in paper https://www.comp.nus.edu.sg/~xiangnan/papers/ncf.pdf
        """
        ratings['rank_latest'] = ratings.groupby(['user_id'])['timestamp'].rank(method='first', ascending=False)
        test = ratings.loc[ratings['rank_latest'] == 1]
        train = ratings.loc[ratings['rank_latest'] > 1]
        assert train['user_id'].nunique()==test['user_id'].nunique(), 'Not Match Train User with Test User'
        return train[['user_id', 'item_id', 'rating']], test[['user_id', 'item_id', 'rating']]


    def _negative_sampling(self, ratings):
        interact_status = (#一个DataFrame
            ratings.groupby('user_id')['item_id']
            .apply(set)#.apply(set)：然后，对于每个用户，它将其交互过的物品ID放入一个集合（set）中，以去除重复的物品。
            .reset_index()#它重置索引，以便得到一个包含用户ID和其交互过的物品ID集合的 DataFrame。
            .rename(columns={'item_id': 'interacted_items'}))
        interact_status['negative_items'] = interact_status['interacted_items'].apply(lambda x: self.item_pool - x)
        #通过从整体物品池 self.item_pool 中减去每个用户的交互物品集合得到的。这意味着 negative_items 包含了每个用户未曾交互的物品。
        interact_status['negative_samples'] = interact_status['negative_items'].apply(lambda x: random.sample(x, self.num_ng_test))
        #negative_samples 的列，其中包含了每个用户的负样本样本。这是通过从 negative_items 中随机抽样 self.num_ng_test 个物品得到的
        return interact_status[['user_id','negative_items', 'negative_samples']]

    def get_train_instance(self):
        users,items, ratings = [], [], []
        train_ratings = pd.merge(self.train_ratings, self.negatives[['user_id', 'negative_items']], on='user_id')
        # 将训练评分数据集 self.train_ratings 与负样本数据 self.negatives[['user_id', 'negative_items']] 进行合并，以获得包含正样本和负样本的数据                                              
        train_ratings['negatives'] = train_ratings['negative_items'].apply(lambda x: random.sample(x, self.num_ng))
        # 创建了一个新的列 negatives，其中包含了每个用户的负样本物品列表，这些负样本物品是从 negative_items 中随机抽样的，数量由 self.num_ng 决定                                              
        for row in train_ratings.itertuples():
            users.append(int(row.user_id))
            items.append(int(row.item_id))
            ratings.append(float(row.rating))
            for i in range(self.num_ng):#添加负样本
                users.append(int(row.user_id))
                items.append(int(row.negatives[i]))
                ratings.append(float(0))  # negative samples get 0 rating
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        
        data = pd.DataFrame({
            'User': users,
            'Item': items,
            'Rating': ratings
        })

        # Save the DataFrame to a CSV file
        data.to_csv('./data/data_div_train_and_test/train_data.csv',header=False,index=False)
        return torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)#num_workers=4改成0，4会发生broken pipe

    def get_test_instance(self):
        users, items, ratings = [], [], []
        test_ratings = pd.merge(self.test_ratings, self.negatives[['user_id', 'negative_samples']], on='user_id')
        for row in test_ratings.itertuples():
            users.append(int(row.user_id))
            items.append(int(row.item_id))
            ratings.append(float(row.rating))
            for i in getattr(row, 'negative_samples'):
                users.append(int(row.user_id))
                items.append(int(i))
                ratings.append(float(0))
                
        data = pd.DataFrame({
            'User': users,
            'Item': items,
            'Rating': ratings
        })
        # Save the DataFrame to a CSV file
        data.to_csv('./data/data_div_train_and_test/test_data.csv',header=False,index=False)
        
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.num_ng_test+1, shuffle=False, num_workers=0)#num_workers=4改成0
    

In [ ]:
# construct the train and test datasets
data = NCF_Data(args, ml_1m)
train_loader =data.get_train_instance()
test_loader =data.get_test_instance()